# 🛒 E-Commerce Demand-Supply System: Exploratory Data Analysis & Spark Demo

This notebook demonstrates how to interact with the **E-Commerce Demand-Supply System** using **Apache Spark (PySpark)**.

### Contents:
1. **Environment Setup & Spark Session**
2. **Data Ingestion & Schema Inspection**
3. **Demand Analysis (Trends, Seasonality, RFM)**
4. **Inventory & Supply Gap Analytics**
5. **Demand Forecasting with Spark MLlib**
6. **Price Elasticity Evaluation**

## 1. Setup Spark Session

In [ ]:
import sys
from pathlib import Path

# Set project root
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config.spark_config import get_spark_session
from pyspark.sql import functions as F

spark = get_spark_session(app_name="EcommerceEDA")
print(f"Spark Version: {spark.version}")

## 2. Ingest and Inspect Cleaned Data

In [ ]:
orders_df = spark.read.parquet("../data/processed/clean_orders")
inventory_df = spark.read.parquet("../data/processed/clean_inventory")

print(f"Total Orders: {orders_df.count():,}")
print(f"Total Inventory Records: {inventory_df.count():,}")
orders_df.printSchema()

## 3. Demand Analysis & Top Selling Categories

In [ ]:
category_kpis = (
    orders_df.groupBy("category")
    .agg(
        F.sum("quantity").alias("total_units_sold"),
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.round(F.avg("unit_price"), 2).alias("avg_selling_price"),
        F.countDistinct("order_id").alias("order_count")
    )
    .orderBy(F.col("total_revenue").desc())
)
category_kpis.show(truncate=False)

## 4. Supply Gap Analysis

In [ ]:
supply_gap_df = spark.read.parquet("../data/processed/supply_gap")
critical_gaps = supply_gap_df.filter(F.col("gap_severity") == "CRITICAL")
print(f"Critical stockout risks detected: {critical_gaps.count()}")
critical_gaps.select("product_id", "category", "total_available", "projected_demand_7d", "supply_gap").show(10)

## 5. Demand Forecast Inspection (MLlib Predictions)

In [ ]:
forecasts_df = spark.read.parquet("../data/processed/demand_forecasts")
forecasts_df.show(15)